# Рекомендация тарифов

В вашем распоряжении данные о поведении клиентов, которые уже перешли на эти тарифы (из проекта курса «Статистический анализ данных»). Нужно построить модель для задачи классификации, которая выберет подходящий тариф. Предобработка данных не понадобится — вы её уже сделали.

Постройте модель с максимально большим значением *accuracy*. Чтобы сдать проект успешно, нужно довести долю правильных ответов по крайней мере до 0.75. Проверьте *accuracy* на тестовой выборке самостоятельно.

## Откройте и изучите файл

1. Добавляем библиотеки
2. Открываем файл и проверяем открывается ли он корректно

In [1]:
from scipy import stats as st
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


In [2]:
try:
    df = pd.read_csv('/datasets/users_behavior.csv')
    df.info()
except:
    print('alarm, волк унес зайчат')

    
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


Ошибок нет, работаем дальше!

## Разбейте данные на выборки

1.Определяем признаки и целевой признак 

2.Делим данные на тренировочную, валидационную и тестовую выборки 60/20/20

In [3]:
df_train, df_valid, df_test = \
              np.split(df.sample(frac=1, random_state=12345), 
                       [int(.6*len(df)), int(.8*len(df))])

In [4]:
features_train = df_train.drop(['is_ultra'], axis=1)
target_train = df_train['is_ultra']
features_valid = df_valid.drop(['is_ultra'], axis=1)
target_valid = df_valid['is_ultra']
test_features = df_test.drop(['is_ultra'], axis=1)
test_target = df_test['is_ultra']

print(df_train.shape)
print(df_valid.shape)
print(df_test.shape)

(1928, 5)
(643, 5)
(643, 5)


## Исследуйте модели

Обучим модели разными методами , заберем из них лучшие результаты


1. Решающее дерево

In [5]:
best_model_tree = None
best_result_tree = 0
for depth in range(1, 10):
    model_tree = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    model_tree.fit(features_train, target_train)
    predictions_tree = model_tree.predict(features_valid)
    result = accuracy_score(target_valid, predictions_tree)
    if result > best_result_tree:
        best_model_tree = model_tree
        best_result_tree = result
        print(f"max_depth = {depth} : {result}")
best_result_tree

max_depth = 1 : 0.7216174183514774
max_depth = 2 : 0.7511664074650077
max_depth = 3 : 0.7667185069984448
max_depth = 4 : 0.7744945567651633
max_depth = 9 : 0.776049766718507


0.776049766718507

2. Случайный лес

In [14]:
best_model_forest = None
best_result_forest = 0
for est in range(1, 10):
    for depth in range(1, 10):
        model_forest = RandomForestClassifier(random_state=12345, n_estimators=est, max_depth = depth)
        model_forest.fit(features_train, target_train)
        predictions_forest = model_forest.predict(features_valid)
        result = accuracy_score(target_valid, predictions_forest)
        if result > best_result_forest:
            best_model_forest = model_forest
            best_result_forest = result
            print(f"max_depth = {est}, {depth} : {result}")
best_result_forest

max_depth = 1, 1 : 0.713841368584759
max_depth = 1, 2 : 0.7511664074650077
max_depth = 1, 5 : 0.7667185069984448
max_depth = 1, 7 : 0.7744945567651633
max_depth = 1, 9 : 0.7807153965785381
max_depth = 4, 8 : 0.7853810264385692
max_depth = 5, 7 : 0.7869362363919129
max_depth = 6, 7 : 0.7916018662519441
max_depth = 9, 7 : 0.7962674961119751


0.7962674961119751

3. Логистическая регрессия

In [7]:
model_logistic = LogisticRegression(random_state=12345,)
model_logistic.fit(features_train, target_train) 
predictions_logistic = model_logistic.predict(features_train)
best_result_logistic = model_logistic.score(features_train, target_train)
best_result_logistic

0.7510373443983402

Наилучший результат из трех моделей показала модель `случайного леса` , самый слабый результат показала модель `логистической регрессии`.

## Проверьте модель на тестовой выборке

1. Решающее дерево

In [20]:
test_predictions_tree = model_tree.predict(test_features)
test_accuracy_tree = accuracy_score(test_target, test_predictions_tree)
print("Обучающая выборка:", best_result_tree)
print("Тестовая выборка:", test_accuracy_tree)

Обучающая выборка: 0.776049766718507
Тестовая выборка: 0.7947122861586314


2. Случайный лес

In [24]:
test_predictions_forest = best_model_forest.predict(test_features)
test_accuracy_forest = accuracy_score(test_target, test_predictions_forest)
print("Обучающая выборка:",best_result_forest)
print("Тестовая выборка:", test_accuracy_forest)

Обучающая выборка: 0.7962674961119751
Тестовая выборка: 0.8009331259720062



3. Логистическая регрессия

In [10]:
test_predictions_logistic = model_logistic.predict(test_features)
test_accuracy_logistic = accuracy_score(test_target, test_predictions_logistic)
print("Обучающая выборка:", best_result_logistic)
print("Тестовая выборка:", test_accuracy_logistic)

Обучающая выборка: 0.7510373443983402
Тестовая выборка: 0.7387247278382582


Лучше всех смогла обучиться модель `случайного леса`, целых 80% правильных ответов

За ней, дыша в спину, идет модель `решающего дерева`, отставая на 0,4%, но при смене гиперпараметров может случитсья переобучение, что ухудшит результат 

Хуже всех себя показала модель `логистической регрессии` всего 74%.

Лучше всех для выбора тарифа подойдет модель `случайного леса`, из-за самого высокого % правильных предсказаний.
